# Case Study 2: Insurance Pricing Risk Model

Audits a claims-risk classifier used to inform premium loading, checking fairness across a synthetic `age_band` attribute — a protected characteristic under the FCA's fair pricing expectations for general insurance. Synthetic data only.

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from responsible_ai_finance import audit

pd.set_option("display.precision", 3)
np.random.seed(42)


In [2]:
rng = np.random.default_rng(11)
n = 1200

driving_experience_years = rng.integers(0, 40, n)
annual_mileage = rng.normal(9000, 3000, n).clip(500, None)
vehicle_value = rng.normal(15000, 6000, n).clip(1000, None)
prior_claims = rng.poisson(0.3, n)
age_band = np.where(driving_experience_years < 5, "Under_25", "25_Plus")

logit = -1.5 + 0.8 * prior_claims - 0.02 * driving_experience_years + annual_mileage / 20000
prob_claim = 1 / (1 + np.exp(-logit))
y = (rng.uniform(0, 1, n) < prob_claim).astype(int)

X = pd.DataFrame(
    {
        "driving_experience_years": driving_experience_years,
        "annual_mileage": annual_mileage,
        "vehicle_value": vehicle_value,
        "prior_claims": prior_claims,
    }
)
X_train, X_test, y_train, y_test, band_train, band_test = train_test_split(
    X, y, age_band, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print(f"Test accuracy: {model.score(X_test, y_test):.3f}")
print(f"Claim rate in test set: {y_test.mean():.3f}")


Test accuracy: 0.753
Claim rate in test set: 0.247


In [3]:
report = audit(
    model,
    X_test.reset_index(drop=True),
    pd.Series(y_test).reset_index(drop=True),
    pd.Series(band_test).reset_index(drop=True),
    model_name="insurance-claim-risk-rf",
)
print(report.to_markdown())


# Responsible AI Audit — insurance-claim-risk-rf
_Generated 2026-09-13T11:37:48.147056+00:00_

## Governance Flags
- ✅ No threshold breaches detected.

## Fairness

- Demographic parity difference: **0.0006**
- Disparate impact ratio: **0.9752** (80% rule: PASS)
- Equalized odds — TPR difference: **0.0444**
- Equalized odds — FPR difference: **0.0169**

| Group | n | Selection rate | TPR | FPR |
|---|---|---|---|---|
| 25_Plus | 314 | 0.022 | 0.039 | 0.017 |
| Under_25 | 46 | 0.022 | 0.083 | 0.000 |

## Explainability

- Top features by mean |SHAP value|: prior_claims, driving_experience_years, vehicle_value, annual_mileage
- SHAP local-fidelity MAE: **0.0000**

## Robustness

- Prediction flip rate under 5% Gaussian noise: **0.5%**
- Most fragile feature to dropout: **driving_experience_years** (7.5% flip rate)


## Key Takeaways

- `driving_experience_years` is used both as a legitimate risk feature and to derive the `age_band` protected attribute — a realistic proxy-variable tension that any fairness audit in insurance pricing has to reason about explicitly, not paper over.
- This case study intentionally does **not** remove the proxy feature; the point of the audit is to surface the tension so a human reviewer can decide, not to silently 'fix' it.